
# Churchland Reach Dataset: Load + First Visualizations

This notebook performs first-pass loading and visualization of the Churchland motor-control NWB file for the 8-direction reach task.

It uses `h5py` directly so it runs even if `pynwb` is not installed.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
from pathlib import Path

sns.set_context("notebook")
sns.set_style("whitegrid")

_candidates = [
    Path("../data/churchland_motor/sub-Jenkins_ses-20090912_behavior+ecephys.nwb"),
    Path("data/churchland_motor/sub-Jenkins_ses-20090912_behavior+ecephys.nwb"),
]
DATA_PATH = next((p for p in _candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not locate Churchland NWB file in ../data or data")
print("Using:", DATA_PATH.resolve())


In [ ]:

# High-level NWB structure preview
with h5py.File(DATA_PATH, "r") as f:
    print("Top-level groups:", list(f.keys()))
    print("Processing modules:", list(f["processing"].keys()))
    print("Behavior signals:", list(f["processing/behavior/Position"].keys()))
    print("Processed ecephys streams:", list(f["processing/ecephys/Processed"].keys()))
    print("Trial count:", f["intervals/trials/id"].shape[0])
    print("Unit count:", f["units/id"].shape[0])


In [ ]:
with h5py.File(DATA_PATH, "r") as f:
    print(f["session_start_time"]))


## Load LFP and Spikes (Churchland)

This cell loads one LFP segment (`A001`/`A002`/`B001`/`B002`) and bins unit spike times into aligned time bins.


In [ ]:


def load_churchland_lfp_spikes(
    nwb_path: Path,
    lfp_series: str = "A001",
    lfp_start_sample: int = 0,
    lfp_num_samples: int = 300_000,
    spike_bin_ms: int = 20,
):
    with h5py.File(nwb_path, "r") as f:
        proc = f["processing/ecephys/Processed"]
        if lfp_series not in proc:
            raise KeyError(f"LFP series '{lfp_series}' not found. Available: {list(proc.keys())}")

        lfp_grp = proc[lfp_series]
        fs_lfp = float(lfp_grp["starting_time"].attrs["rate"])
        t0 = float(lfp_grp["starting_time"][()])

        n_total = int(lfp_grp["data"].shape[0])
        lfp_end_sample = n_total if lfp_num_samples is None else min(n_total, lfp_start_sample + int(lfp_num_samples))
        if lfp_start_sample < 0 or lfp_start_sample >= lfp_end_sample:
            raise ValueError("Invalid LFP sample range")

        lfp_raw = lfp_grp["data"][lfp_start_sample:lfp_end_sample, :]
        lfp_time_s = t0 + np.arange(lfp_start_sample, lfp_end_sample) / fs_lfp

        spike_times_all = f["units/spike_times"][:]
        spike_idx = f["units/spike_times_index"][:]
        unit_ids = f["units/id"][:]

    starts = np.concatenate([[0], spike_idx[:-1]])
    stops = spike_idx

    dt = spike_bin_ms / 1000.0
    t_start = float(lfp_time_s[0])
    t_stop = float(lfp_time_s[-1]) + 1.0 / fs_lfp
    bin_edges = np.arange(t_start, t_stop + dt, dt)

    n_bins = len(bin_edges) - 1
    n_units = len(unit_ids)
    spike_count = np.zeros((n_bins, n_units), dtype=np.int32)

    for u in range(n_units):
        st = spike_times_all[starts[u]:stops[u]]
        m = (st >= t_start) & (st < t_stop)
        if np.any(m):
            spike_count[:, u], _ = np.histogram(st[m], bins=bin_edges)

    spike_rate = spike_count.astype(np.float32) / dt

    return {
        "lfp_raw": lfp_raw,
        "fs_lfp": fs_lfp,
        "lfp_time_s": lfp_time_s,
        "spike_count": spike_count,
        "spike_rate": spike_rate,
        "spike_bin_edges_s": bin_edges,
        "spike_bin_ms": spike_bin_ms,
        "unit_ids": unit_ids,
        "lfp_series": lfp_series,
    }

churchland = load_churchland_lfp_spikes(
    DATA_PATH,
    lfp_series="A001",       # A001, A002, B001, B002
    lfp_start_sample=0,
    lfp_num_samples=300_000,
    spike_bin_ms=20,
)

lfp_raw = churchland["lfp_raw"]
spike_count = churchland["spike_count"]
spike_rate = churchland["spike_rate"]
unit_ids = churchland["unit_ids"]

print("Loaded Churchland LFP + spikes")
print("lfp_raw:", lfp_raw.shape, "fs_lfp:", churchland["fs_lfp"], "series:", churchland["lfp_series"])
print("spike_count:", spike_count.shape, "(bins x units)")
print("spike_rate:", spike_rate.shape)
print("n_units:", unit_ids.shape[0])


In [ ]:

def load_trials_dataframe(nwb_path: Path) -> pd.DataFrame:
    with h5py.File(nwb_path, "r") as f:
        tr = f["intervals/trials"]
        df = pd.DataFrame({
            "trial_id": tr["id"][:],
            "start_time": tr["start_time"][:],
            "stop_time": tr["stop_time"][:],
            "go_cue_time": tr["go_cue_time"][:],
            "move_begins_time": tr["move_begins_time"][:],
            "move_ends_time": tr["move_ends_time"][:],
            "reaction_time": tr["reaction_time"][:],
            "correct_reach": tr["correct_reach"][:].astype(bool),
            "task_success": tr["task_success"][:].astype(bool),
            "discard_trial": tr["discard_trial"][:].astype(bool),
            "trial_type": tr["trial_type"][:],
            "maze_condition": tr["maze_condition"][:],
        })
        hit_xy = tr["hit_target_position"][:]  # (n_trials, 2)

    df["target_x"] = hit_xy[:, 0]
    df["target_y"] = hit_xy[:, 1]
    df["target_angle_rad"] = np.arctan2(df["target_y"], df["target_x"])
    df["target_angle_deg"] = (np.degrees(df["target_angle_rad"]) + 360) % 360
    df["target_bin_45deg"] = (np.floor(df["target_angle_deg"] / 45) % 8).astype(int)
    return df

trials_df = load_trials_dataframe(DATA_PATH)
trials_df.head()


In [ ]:

print("Trials:", len(trials_df))
print("Valid (not discarded):", (~trials_df["discard_trial"]).sum())
print("Successful:", trials_df["task_success"].sum())
print("Correct reach:", trials_df["correct_reach"].sum())
print("Median reaction time (s):", np.nanmedian(trials_df["reaction_time"]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(trials_df["reaction_time"].dropna(), bins=40, ax=axes[0], color="#4c72b0")
axes[0].set_title("Reaction time distribution")
axes[0].set_xlabel("Reaction time (s)")

counts = trials_df["target_bin_45deg"].value_counts().sort_index()
axes[1].bar(counts.index, counts.values, color="#dd8452")
axes[1].set_xticks(range(8))
axes[1].set_title("Target direction bins (45 deg)")
axes[1].set_xlabel("Direction bin")
axes[1].set_ylabel("Trial count")

plt.tight_layout()


In [ ]:

def plot_trial_trajectories(nwb_path: Path, trials: pd.DataFrame, n_trials: int = 30, signal: str = "Hand"):
    valid = trials.loc[(~trials["discard_trial"]) & trials["task_success"]].copy()
    sample = valid.sample(min(n_trials, len(valid)), random_state=0).sort_values("start_time")

    with h5py.File(nwb_path, "r") as f:
        pos_grp = f[f"processing/behavior/Position/{signal}"]
        ts = pos_grp["timestamps"]
        xy = pos_grp["data"]

        plt.figure(figsize=(7, 7))
        for _, tr in sample.iterrows():
            i0 = np.searchsorted(ts, tr["start_time"], side="left")
            i1 = np.searchsorted(ts, tr["stop_time"], side="right")
            seg = xy[i0:i1]
            if len(seg) < 2:
                continue
            plt.plot(seg[:, 0], seg[:, 1], alpha=0.35, lw=1)

    plt.title(f"{signal} trajectories across sampled successful trials")
    plt.xlabel("X position")
    plt.ylabel("Y position")
    plt.axis("equal")
    plt.show()

plot_trial_trajectories(DATA_PATH, trials_df, n_trials=40, signal="Hand")


In [ ]:

def unit_firing_rate_summary(nwb_path: Path):
    with h5py.File(nwb_path, "r") as f:
        spike_idx = f["units/spike_times_index"][:]
        unit_ids = f["units/id"][:]
        session_end = float(f["intervals/trials/stop_time"][:].max())

    counts = np.diff(np.concatenate([[0], spike_idx]))
    rates = counts / session_end

    summary = pd.DataFrame({
        "unit_id": unit_ids,
        "spike_count": counts,
        "rate_hz": rates,
    })
    return summary

rates_df = unit_firing_rate_summary(DATA_PATH)
print(rates_df.describe())

plt.figure(figsize=(7, 4))
sns.histplot(rates_df["rate_hz"], bins=40, color="#55a868")
plt.title("Single-unit mean firing rates")
plt.xlabel("Rate (Hz)")
plt.ylabel("Unit count")
plt.tight_layout()


In [ ]:

def plot_spike_raster_for_trial(nwb_path: Path, trials: pd.DataFrame, trial_idx: int = 0, n_units: int = 30):
    valid = trials.loc[(~trials["discard_trial"]) & trials["task_success"]].reset_index(drop=True)
    if trial_idx >= len(valid):
        raise ValueError(f"trial_idx {trial_idx} out of range for {len(valid)} valid trials")
    tr = valid.iloc[trial_idx]

    with h5py.File(nwb_path, "r") as f:
        spk = f["units/spike_times"]
        idx = f["units/spike_times_index"][:]
        unit_ids = f["units/id"][:]

        starts = np.concatenate([[0], idx[:-1]])
        stops = idx

        order = np.argsort(stops - starts)[::-1]
        chosen = order[: min(n_units, len(order))]

        plt.figure(figsize=(10, 6))
        for row, u in enumerate(chosen):
            u_spk = spk[starts[u]:stops[u]]
            m = (u_spk >= tr["start_time"]) & (u_spk <= tr["stop_time"])
            x = u_spk[m] - tr["go_cue_time"]
            if len(x) > 0:
                plt.vlines(x, row + 0.6, row + 1.4, color="black", linewidth=0.7)

    plt.axvline(0.0, color="red", linestyle="--", linewidth=1, label="Go cue")
    plt.xlabel("Time from go cue (s)")
    plt.ylabel("Units (sorted by overall spike count)")
    plt.title(f"Spike raster around one successful trial (trial_id={int(tr['trial_id'])})")
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

plot_spike_raster_for_trial(DATA_PATH, trials_df, trial_idx=5, n_units=40)



## Next Step for Pipeline

Once these checks look good, the next pipeline cell should time-bin spikes and behavior onto a shared grid (for example 10-20 ms bins) and align each trial to `go_cue_time` or `move_begins_time`.



## Compact Modeling Pipeline

Prepare aligned spike/LFP/behavior data, filter LFP to beta band, train the 8 embeddings, and show only:
- all-trials embeddings
- direction-averaged embeddings


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
import sys

for p_ in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    code_dir = p_ / "code"
    if (code_dir / "comets").exists():
        if str(code_dir) not in sys.path:
            sys.path.insert(0, str(code_dir))
        break

import comets
from comets.encoders import CEBRASKipEncoder, SkippedConvEncoderConfig
from comets.configs import ModalityConfig, SamplingConfig, TrainingConfig
from comets.heads import LinearHead

try:
    import cebra
except ImportError as e:
    raise ImportError("Please install CEBRA in this kernel: pip install cebra") from e


In [ ]:
# Optional: print explicit NWB warning about cross-array sync
with h5py.File(DATA_PATH, "r") as f:
    for key in ["A001", "A002", "B001", "B002"]:
        path = f"processing/ecephys/Processed/{key}"
        desc = f[path].attrs.get("description", "")
        if isinstance(desc, bytes):
            desc = desc.decode("utf-8", errors="ignore")
        print(f"\n{path}\n{desc}")


In [ ]:

# Prepare aligned dataset (same style as before)

def downsample_lfp_to_bins(lfp_raw, fs_lfp, bin_ms):
    step = int(round(fs_lfp * (bin_ms / 1000.0)))
    n = (lfp_raw.shape[0] // step) * step
    x = lfp_raw[:n]
    return x.reshape(n // step, step, lfp_raw.shape[1]).mean(axis=1)


def interpolate_behavior_hand(nwb_path, t_centers):
    with h5py.File(nwb_path, "r") as f:
        hand_ts = f["processing/behavior/Position/Hand/timestamps"][:]
        hand_xy = f["processing/behavior/Position/Hand/data"][:]
    bx = np.interp(t_centers, hand_ts, hand_xy[:, 0])
    by = np.interp(t_centers, hand_ts, hand_xy[:, 1])
    return np.column_stack([bx, by])


def assign_direction_labels(t_centers, trials):
    labels = np.full(t_centers.shape[0], -1, dtype=np.int32)
    valid_trials = trials.loc[(~trials["discard_trial"]) & trials["task_success"]].copy()
    for _, tr in valid_trials.iterrows():
        i0 = np.searchsorted(t_centers, tr["start_time"], side="left")
        i1 = np.searchsorted(t_centers, tr["stop_time"], side="right")
        labels[i0:i1] = int(tr["target_bin_45deg"])
    return labels


def zscore_channels(X):
    return (X - X.mean(axis=0, keepdims=True)) / (X.std(axis=0, keepdims=True) + 1e-6)


spike_edges = churchland["spike_bin_edges_s"]
t_centers = 0.5 * (spike_edges[:-1] + spike_edges[1:])

spike_rate_full = churchland["spike_rate"].astype(np.float32)
lfp_raw_full = churchland["lfp_raw"].astype(np.float32)
fs_lfp = float(churchland["fs_lfp"])
bin_ms = int(churchland["spike_bin_ms"])

# raw lfp on spike-bin grid
lfp_raw_bin = downsample_lfp_to_bins(lfp_raw_full, fs_lfp, bin_ms).astype(np.float32)

# beta lfp on spike-bin grid (13-30 Hz)
nyq = 0.5 * fs_lfp
b_beta, a_beta = butter(4, [13.0 / nyq, 30.0 / nyq], btype="band")
lfp_beta_full = filtfilt(b_beta, a_beta, lfp_raw_full, axis=0)
lfp_beta_bin = downsample_lfp_to_bins(lfp_beta_full, fs_lfp, bin_ms).astype(np.float32)

# gamma lfp on spike-bin grid (30-150 Hz)
nyq = 0.5 * fs_lfp
b_gamma, a_gamma = butter(4, [30.0 / nyq, 150.0 / nyq], btype="band")
lfp_gamma_full = filtfilt(b_gamma, a_gamma, lfp_raw_full, axis=0)
lfp_gamma_bin = downsample_lfp_to_bins(lfp_gamma_full, fs_lfp, bin_ms).astype(np.float32)

lfp_filtered_bin = lfp_gamma_bin + lfp_beta_bin

n = min(len(t_centers), len(spike_rate_full), len(lfp_raw_bin), len(lfp_filtered_bin))
t_centers = t_centers[:n]
spike_rate_full = spike_rate_full[:n]
lfp_raw_bin = lfp_raw_bin[:n]
lfp_filtered_bin = lfp_filtered_bin[:n]

behavior_xy = interpolate_behavior_hand(DATA_PATH, t_centers)
dir_label = assign_direction_labels(t_centers, trials_df)
mask = (dir_label >= 0) & np.isfinite(behavior_xy).all(axis=1)

A_all = zscore_channels(spike_rate_full[mask])
B_raw_all = zscore_channels(lfp_raw_bin[mask])
B_filtered_all = zscore_channels(lfp_filtered_bin[mask])
behavior_all = behavior_xy[mask]
dir_all = dir_label[mask]
time_all = t_centers[mask]

print("A_all:", A_all.shape)
print("B_raw_all:", B_raw_all.shape)
print("B_filtered_all:", B_filtered_all.shape)
print("behavior_all:", behavior_all.shape)
print("dirs:", np.unique(dir_all), "counts:", {int(d): int((dir_all==d).sum()) for d in np.unique(dir_all)})


In [ ]:

# Train the requested 8 embeddings

FAST_MODE = False  # set False for longer training
cebra_iterations = 80 if FAST_MODE else 500
comets_epochs = 50 if FAST_MODE else 120

# CEBRA-time / CEBRA-beh on spike and lfp
cebra_time_spike = cebra.CEBRA(model_architecture="offset10-model", max_iterations=cebra_iterations,
                               output_dimension=3, batch_size=256, temperature=1.0, verbose=True)
Z_cebra_time_spike = cebra_time_spike.fit_transform(A_all)

cebra_time_lfp = cebra.CEBRA(model_architecture="offset10-model", max_iterations=cebra_iterations,
                             output_dimension=3, batch_size=256, temperature=1.0, verbose=True)
Z_cebra_time_lfp = cebra_time_lfp.fit_transform(B_raw_all)

cebra_beh_spike = cebra.CEBRA(model_architecture="offset10-model", max_iterations=cebra_iterations,
                              output_dimension=3, batch_size=256, temperature=1.0, verbose=True)
Z_cebra_beh_spike = cebra_beh_spike.fit_transform(A_all, behavior_all)

cebra_beh_lfp = cebra.CEBRA(model_architecture="offset10-model", max_iterations=cebra_iterations,
                            output_dimension=3, batch_size=256, temperature=1.0, verbose=True)
Z_cebra_beh_lfp = cebra_beh_lfp.fit_transform(B_raw_all, behavior_all)

# CoMETs config
modA = ModalityConfig(
    encoder_cls=CEBRASKipEncoder,
    encoder_cfg=SkippedConvEncoderConfig(in_channels=A_all.shape[1], emb_dim=3),
    win_len=8,
)
modB_raw = ModalityConfig(
    encoder_cls=CEBRASKipEncoder,
    encoder_cfg=SkippedConvEncoderConfig(in_channels=B_raw_all.shape[1], emb_dim=3),
    win_len=8,
)
modB_beta = ModalityConfig(
    encoder_cls=CEBRASKipEncoder,
    encoder_cfg=SkippedConvEncoderConfig(in_channels=B_filtered_all.shape[1], emb_dim=3),
    win_len=8,
)
sampling = SamplingConfig(batch_size=256, pos_offset=5, negative_mask_dist=16)
train_cfg = TrainingConfig(epochs=comets_epochs, lr=1e-3, temperature=1.0,
                           proj_dim=3, alpha_cross=0.1, beta=0.8, head_cls=LinearHead)

# CoMETs raw
comets_raw = comets.train(A_all, modA, B_raw_all, modB_raw, sampling, train_cfg)
Z_comets_raw_spike = comets_raw.embedA(A_all)
Z_comets_raw_lfp = comets_raw.embedB(B_raw_all)

# CoMETs beta
comets_beta = comets.train(A_all, modA, B_filtered_all, modB_beta, sampling, train_cfg)
Z_comets_beta_spike = comets_beta.embedA(A_all)
Z_comets_beta_lfp = comets_beta.embedB(B_filtered_all)

EMBEDDINGS = {
    "cebra-time spike": Z_cebra_time_spike,
    "cebra-time lfp": Z_cebra_time_lfp,
    "cebra-beh spike": Z_cebra_beh_spike,
    "cebra-beh lfp": Z_cebra_beh_lfp,
    "comets-raw spike": Z_comets_raw_spike,
    "comets-raw lfp": Z_comets_raw_lfp,
    "comets-beta spike": Z_comets_beta_spike,
    "comets-beta lfp": Z_comets_beta_lfp,
}

for k, v in EMBEDDINGS.items():
    print(f"{k:18s}: {v.shape}")


In [ ]:

# Visualization helper

def first3(Z):
    Z = np.asarray(Z)
    if Z.shape[1] < 3:
        Z = np.concatenate([Z, np.zeros((Z.shape[0], 3 - Z.shape[1]))], axis=1)
    return Z[:, 0], Z[:, 1], Z[:, 2]


def direction_averaged_trajectories(Z, t, trials, pre_bins=10, post_bins=25):
    Z = np.asarray(Z)
    valid_trials = trials.loc[(~trials["discard_trial"]) & trials["task_success"]].copy()
    by_dir = {d: [] for d in range(8)}

    for _, tr in valid_trials.iterrows():
        c = np.searchsorted(t, float(tr["go_cue_time"]))
        s = c - pre_bins
        e = c + post_bins
        if s < 0 or e > len(t):
            continue
        d = int(tr["target_bin_45deg"])
        by_dir[d].append(Z[s:e])

    avg, cnt = {}, {}
    for d in range(8):
        cnt[d] = len(by_dir[d])
        avg[d] = np.mean(np.stack(by_dir[d], axis=0), axis=0) if cnt[d] > 0 else None
    return avg, cnt


In [ ]:

# 1) General all-trials embeddings

fig = plt.figure(figsize=(10, 8))
for i, (name, Z) in enumerate(EMBEDDINGS.items(), start=1):
    ax = fig.add_subplot(4, 2, i, projection="3d")
    x, y, z = first3(Z)
    ax.scatter(x, y, z, c=dir_all, cmap="tab10", s=2, alpha=0.7)
    ax.set_title(name, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])

fig.suptitle("All-trials embeddings", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:

# 2) Direction-averaged embeddings

fig = plt.figure(figsize=(10, 8))
for i, (name, Z) in enumerate(EMBEDDINGS.items(), start=1):
    ax = fig.add_subplot(4, 2, i, projection="3d")
    cmap = plt.get_cmap("tab10")
    avg, cnt = direction_averaged_trajectories(Z, time_all, trials_df)

    for d in range(8):
        traj = avg[d]
        if traj is None:
            continue
        x, y, z = first3(traj)
        color = cmap(d)
        ax.plot(x, y, z, color=color, lw=1.8)
        ax.scatter([x[0]], [y[0]], [z[0]], color=color, s=14, marker="o")
        ax.scatter([x[-1]], [y[-1]], [z[-1]], color=color, s=14, marker="^")

    ax.set_title(name, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])

fig.suptitle("Direction-averaged embeddings", fontsize=14)
plt.tight_layout()
plt.show()
